<a href="https://colab.research.google.com/github/SAKURA-hub69/-/blob/main/%E5%A4%A7%E5%95%8F%E9%9B%A3%E6%98%93%E5%BA%A6%E5%88%86%E6%9E%90_ops20_ipynb%EF%BC%88%E8%8B%B1%E8%AA%9E%E7%A7%91%EF%BC%89.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
from google import colab

colab.drive.mount('/content/gdrive')
#ディレクトリをプログラムが存在する演習フォルダに移動
%cd '/content/gdrive/My Drive/復習の指針/analysis'

kyoka_id = 1
input_path = "./OPSTTS_DAIMON_SEITO_KEKKA_2020_kyoka_id_1.csv"

print("試験種の講座コード(4桁)+科目コード(2桁)を入力してください。複数入力の場合は半角スペースをあけてください。\n英語の科目コードは10なので、入力は****10となります。\n東大文系英語なら140810とか。東大文系と東大理系は講座コード1408で統一してあります。")
KODE_list = list(input().split())

#データ読み込み
df = pd.read_csv(input_path, dtype=str, usecols=["KOZA_CODE","SHIKEN_ID","SHIKEN_NENDO","HAITEN","TOKUTEN","DAIMON_ID"])
df['JUKEN_KAMOKU_ID'] = df['SHIKEN_ID'].str[4:6]
#１回目答案でもともと絞られているぽい
df.head()
print('読み込み完了')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
/content/gdrive/My Drive/復習の指針/analysis
試験種の講座コード(4桁)+科目コード(2桁)を入力してください。複数入力の場合は半角スペースをあけてください。
英語の科目コードは10なので、入力は****10となります。
東大文系英語なら140810とか。東大文系と東大理系は講座コード1408で統一してあります。


KeyboardInterrupt: ignored

In [ ]:
for KODE in KODE_list:
  df_out = pd.DataFrame()
  KOZA_KODE = KODE[:4]
  JUKEN_KAMOKU_ID = KODE[4:]
  output_path = './ダウンロード/daimon_analyze_ops20_' + KODE + '.csv'

  for nendo in range(2020, 2010, -1):
    nendo_var = 2020
    nendo = str(nendo)
    df_extract = df.query('KOZA_CODE == @KOZA_KODE & JUKEN_KAMOKU_ID == @JUKEN_KAMOKU_ID & SHIKEN_NENDO == @nendo')
    df_extract = df_extract.astype({'HAITEN':'int64', 'TOKUTEN':'int64'})

    #info = df_extract[['TOKUTEN','DAIMON_ID']].groupby('DAIMON_ID').describe()
    df_sample = pd.DataFrame()
    for i in range(0, 105, 5):
      i /= 100
      info_1 = df_extract[['TOKUTEN','DAIMON_ID']].groupby('DAIMON_ID').quantile(i)
      info_1.rename(columns={'TOKUTEN': f"{(1-i) * 100:.0f}%"}, inplace=True)
      df_sample = pd.concat([df_sample, info_1], axis=1)

    info = df_extract[['TOKUTEN','DAIMON_ID']].groupby('DAIMON_ID').describe()
    info = info['TOKUTEN'][['count', 'mean', 'std']]
    info = pd.concat([info, df_sample],axis=1)
    info.insert(0,'NENDO', nendo)

    df_out = pd.concat([df_out,info],axis=0)
    nendo_var -= 1
  df_out.to_csv(output_path, encoding='utf-8')
  files.download(output_path)
print('ファイルダウンロードが完了しました。')#全てダウンロードされない場合があるので、その時はダウンロードフォルダから手動でお願いします。

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ファイルダウンロードが完了しました。


In [ ]:
#全部合わせたファイルがほしいときはこっち
id = list(input().split())
df_ans = pd.DataFrame()
for i in id:
  output_paths = './ダウンロード/daimon_analyze_ops20_' + str(i) + '.csv'
  df = pd.read_csv(output_paths)
  df.insert(0, 'NUMBER',i)
  df_ans = pd.concat([df_ans, df], axis=0)

df_ans.head()
df_ans.to_csv('ダウンロード/daimon.csv')
files.download('ダウンロード/daimon.csv')

140810


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>